In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="02-visual-search",
    notebook_name="02_embedding_and_contrastive_learning.ipynb"
)

# Embedding Spaces & Contrastive Learning Deep Dive

## What This Notebook Covers

In Notebook 01, we designed the full visual search system. Now we zoom in on the **brain** of the system -- the part that learns what makes two images "similar."

Think of it like this: imagine you have a magical ruler that can measure how similar any two photos are. This notebook teaches you how to **build that ruler** from scratch.

We cover:

1. Embedding spaces -- what they are and how to visualize them
2. Contrastive learning frameworks (SimCLR, MoCo)
3. How to create positive pairs (human judgment, clicks, self-supervision)
4. NT-Xent loss -- derivation and code
5. Temperature parameter -- how it shapes training
6. Hard negative mining -- making training harder (in a good way)
7. Fine-tuning a pre-trained model
8. Evaluating embedding quality

---

## 1. Embedding Spaces: The Map of All Images

### The Analogy

Think of every image as a student in a giant school cafeteria. At lunchtime, students naturally sit with people who are similar to them -- the soccer players sit together, the chess club sits together, the band kids sit together.

An **embedding space** is like the seating chart of that cafeteria. Every image gets a "seat" (a list of numbers called a **vector**), and similar images end up sitting near each other.

```
Embedding Space (simplified to 2D):

         Dogs cluster
    *  *                Cars cluster
      *  *                  o  o
    *                        o  o
                                o
  Food cluster
    +  +  +
      +  +
```

### Staff-Level Detail

An embedding is a **dense, continuous, fixed-dimensional vector** representation of an input. For images:
- Input: 224x224x3 = 150,528 raw pixel values
- Output: 256-dimensional embedding vector
- The model compresses the image into a compact "fingerprint"
- L2-normalized embeddings live on a **unit hypersphere**, making cosine similarity = dot product

Let's visualize this!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.datasets import make_blobs
import torch
import torch.nn as nn
import torch.nn.functional as F

# ======================================================================
# Simulating an Embedding Space
#
# In real life, we'd extract embeddings from a trained CNN.
# Here we simulate 5 image categories with clusters in 256-D space,
# then use t-SNE to project down to 2D for visualization.
# ======================================================================

np.random.seed(42)

# Simulate 5 categories: dogs, cars, food, buildings, flowers
categories = ['Dogs', 'Cars', 'Food', 'Buildings', 'Flowers']
colors = ['#E53935', '#1E88E5', '#43A047', '#FB8C00', '#8E24AA']
n_per_category = 100
embedding_dim = 256

# Create cluster centers in 256-D space
# (In reality, the model LEARNS these clusters during training)
all_embeddings = []
all_labels = []

for i, category in enumerate(categories):
    # Random center for each category
    center = np.random.randn(embedding_dim) * 2
    # Points clustered around the center (with some spread)
    points = center + np.random.randn(n_per_category, embedding_dim) * 0.5
    # L2 normalize (like our model does)
    norms = np.linalg.norm(points, axis=1, keepdims=True)
    points = points / norms
    all_embeddings.append(points)
    all_labels.extend([i] * n_per_category)

all_embeddings = np.vstack(all_embeddings)
all_labels = np.array(all_labels)

# ---- t-SNE projection from 256-D to 2-D ----
print("Running t-SNE to project 256-D embeddings to 2-D for visualization...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(all_embeddings)

# ---- Plot ----
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

for i, category in enumerate(categories):
    mask = all_labels == i
    ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
               c=colors[i], label=category, alpha=0.6, s=40, edgecolors='white', linewidth=0.5)

ax.set_title('Embedding Space Visualization (t-SNE Projection)', fontsize=16, fontweight='bold')
ax.legend(fontsize=12, loc='upper right')
ax.set_xlabel('t-SNE dimension 1', fontsize=12)
ax.set_ylabel('t-SNE dimension 2', fontsize=12)
ax.grid(True, alpha=0.3)

# Add annotation
ax.text(0.02, 0.02, 'Each dot = one image embedding (256-D projected to 2-D)\n'
        'Similar images cluster together!',
        transform=ax.transAxes, fontsize=10, style='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  - Each category forms a tight cluster")
print("  - Similar categories might overlap slightly (that's expected)")
print("  - The model's job is to LEARN these clusters during training")
print(f"  - Original embedding dimension: {embedding_dim}")
print(f"  - Number of images: {len(all_labels)}")

In [ ]:
# ======================================================================
# Measuring Similarity Between Embeddings
#
# With L2-normalized embeddings, cosine similarity = dot product.
# This is why we L2-normalize: it makes similarity computation trivial.
# ======================================================================

# Pick one "query" image from the Dogs category
query_idx = 0  # first dog
query_emb = all_embeddings[query_idx]

# Compute cosine similarity to ALL other images
# Since embeddings are L2-normalized: cosine_sim = dot product
similarities = all_embeddings @ query_emb  # (500,) dot products

# Top 5 most similar (excluding self)
top_indices = np.argsort(similarities)[::-1][1:6]  # skip self

print(f"Query image: {categories[all_labels[query_idx]]} (index {query_idx})")
print(f"\nTop 5 most similar images:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  #{rank}: {categories[all_labels[idx]]} (similarity = {similarities[idx]:.4f})")

# Bottom 3 least similar
bottom_indices = np.argsort(similarities)[:3]
print(f"\nBottom 3 least similar images:")
for rank, idx in enumerate(bottom_indices, 1):
    print(f"  #{rank}: {categories[all_labels[idx]]} (similarity = {similarities[idx]:.4f})")

print("\n--- Interview Key Insight ---")
print("Cosine similarity ranges from -1 to +1 for L2-normalized vectors.")
print("In practice, most similarities are between 0 and 1 for image embeddings.")
print("We prefer cosine/dot-product over Euclidean distance because of the")
print("curse of dimensionality -- in high-D space, all Euclidean distances")
print("converge to similar values, making them less discriminative.")

## 2. Contrastive Learning Frameworks

### The Core Idea (for a 12-year-old)

Imagine you're training a puppy to recognize things. You show the puppy two photos of the same dog and say "these are the SAME!" Then you show a dog and a cat and say "these are DIFFERENT!" Over thousands of examples, the puppy learns what makes images similar.

That's exactly what contrastive learning does with a neural network.

### SimCLR (Simple Contrastive Learning of Representations)

**Created by:** Google Brain (2020)

**Key idea:** Take one image, augment it two different ways, and train the model to know they're the same image.

```
SimCLR Pipeline:

                    [Random Crop + Color Jitter]
Original Image ---->  Augmented View 1  ----> Encoder ----> z_1
       |                                                      |
       |            [Random Crop + Flip]                      |  Maximize
       +----------->  Augmented View 2  ----> Encoder ----> z_2  Similarity
                                                              |
                    All other images in batch = Negatives  <---+
```

### MoCo (Momentum Contrast)

**Created by:** Meta AI (2019)

**Key idea:** Uses a "memory bank" of past embeddings as negatives, so you get MORE negatives without needing a huge batch size.

```
MoCo Pipeline:

Query Image ----> Encoder (updated by gradient)   ----> q
                                                          |
Key Image   ----> Momentum Encoder (slow-moving copy) --> k
                                                          |
Queue of past keys (dynamic dictionary)  <----- enqueue k
                                         -----> dequeue oldest
```

### SimCLR vs MoCo: When to Use What

| Feature | SimCLR | MoCo |
|---------|--------|------|
| Negatives from | Same batch | Memory queue |
| Batch size needed | Very large (4096+) | Small is fine (256) |
| GPU memory | High | Moderate |
| Implementation | Simpler | More complex |
| Performance | Excellent | Excellent |
| Best for | When you have many GPUs | When GPU memory is limited |

**Interview tip:** Mention both, say you'd start with SimCLR for simplicity, and switch to MoCo if GPU memory is a bottleneck.

In [ ]:
# ======================================================================
# SimCLR: Positive Pair Generation via Data Augmentation
#
# The MOST important ingredient in SimCLR is the augmentation pipeline.
# Different augmentations of the same image = positive pair.
# All other images in the batch = negative pairs.
# ======================================================================

import torchvision.transforms as transforms
from PIL import Image

# SimCLR augmentation pipeline (following the original paper)
class SimCLRAugmentation:
    """
    Creates two augmented views of the same image.
    
    Think of it like taking two different photos of the same scene:
    - One photo is zoomed in and slightly brighter
    - The other photo is zoomed out and in grayscale
    The model should know they're the same scene!
    """
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.RandomResizedCrop(size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.GaussianBlur(kernel_size=23, sigma=(0.1, 2.0)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    
    def __call__(self, image):
        """Returns two different augmented views of the same image."""
        return self.transform(image), self.transform(image)


# Demo: show different augmentations of a single image
# We'll create a simple gradient image to visualize the augmentations
dummy_img = np.zeros((300, 300, 3), dtype=np.uint8)
# Create a colorful pattern
for i in range(300):
    for j in range(300):
        dummy_img[i, j, 0] = int(255 * i / 300)   # Red gradient
        dummy_img[i, j, 1] = int(255 * j / 300)   # Green gradient
        dummy_img[i, j, 2] = 128                    # Constant blue

pil_img = Image.fromarray(dummy_img)

# Visualize-friendly augmentation (no normalization, for display)
viz_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)
    ], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
])

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(dummy_img)
axes[0].set_title('Original', fontsize=12, fontweight='bold')

for i in range(1, 5):
    augmented = viz_transform(pil_img)
    axes[i].imshow(augmented.permute(1, 2, 0).clamp(0, 1))
    axes[i].set_title(f'Augmented View {i}', fontsize=12, fontweight='bold')

for ax in axes:
    ax.axis('off')

fig.suptitle('SimCLR Augmentations: Same Image, Different Views', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each augmentation randomly applies:")
print("  - Random crop and resize (different regions)")
print("  - Random horizontal flip")
print("  - Color jitter (brightness, contrast, saturation, hue)")
print("  - Random grayscale conversion (20% chance)")
print("  - Gaussian blur")
print("\nThe model must learn: 'These all come from the SAME image!'")

## 3. Positive Pair Generation Strategies

### Three Methods Compared

Before training, we need **pairs** of images that are "similar." There are three ways to get them:

| Method | How It Works | Like... | Pros | Cons |
|--------|-------------|---------|------|------|
| **Human Judgment** | Pay annotators to find similar images | Hiring librarians to sort books | Most accurate | Expensive, slow, doesn't scale |
| **User Clicks** | User clicked B after searching A = similar | Watching which books people browse together | Automatic, reflects real user needs | Noisy (random clicks), sparse |
| **Self-Supervision** | Augment image A to create A' = positive pair | Making photocopies with slight changes | Free, scalable, not noisy | Augmented != truly similar images |

### Our Choice: Self-Supervision

Why? Because we have **200 billion images**. We can't annotate that. Self-supervision is free, scalable, and frameworks like SimCLR have proven it works. We can always fine-tune with click data later.

**Staff-level nuance:** In practice, the best systems use a **hybrid approach**:
1. Start with self-supervised pre-training (SimCLR/MoCo)
2. Fine-tune with click data (noisy but real signals)
3. Use human annotations for evaluation datasets only

In [ ]:
# ======================================================================
# Simulating All Three Positive Pair Strategies
# ======================================================================

import pandas as pd

# Strategy 1: Human Judgment
print("=" * 70)
print("STRATEGY 1: Human Judgment")
print("=" * 70)
human_pairs = pd.DataFrame({
    'query_image': ['golden_retriever_01.jpg', 'red_car_07.jpg', 'pizza_42.jpg'],
    'positive_image': ['golden_retriever_09.jpg', 'red_sedan_03.jpg', 'margherita_11.jpg'],
    'similarity_score': [4.8, 4.2, 4.5],
    'annotator': ['annotator_A', 'annotator_B', 'annotator_A'],
    'cost_per_pair': ['$0.10', '$0.10', '$0.10']
})
print(human_pairs.to_string(index=False))
print(f"\nFor 1M pairs: cost = $100,000, time = weeks")
print(f"For 1B pairs: cost = $100,000,000 -- NOT scalable!")

# Strategy 2: Click Data
print("\n" + "=" * 70)
print("STRATEGY 2: User Click Data")
print("=" * 70)
click_pairs = pd.DataFrame({
    'user_id': [42, 42, 99, 101, 101],
    'query_image': ['sneaker_01', 'sneaker_01', 'lamp_77', 'cat_05', 'cat_05'],
    'clicked_image': ['sneaker_09', 'boot_33', 'lamp_82', 'cat_12', 'dog_01'],
    'is_truly_similar': [True, False, True, True, False],
    'note': ['Good signal', 'NOISY -- boot != sneaker', 'Good signal', 
             'Good signal', 'NOISY -- misclick']
})
print(click_pairs.to_string(index=False))
print(f"\nNoise rate in practice: ~20-40% of clicks are not truly similar")
print(f"Sparsity problem: most images have 0 clicks")

# Strategy 3: Self-Supervision
print("\n" + "=" * 70)
print("STRATEGY 3: Self-Supervision (SimCLR)")
print("=" * 70)
self_sup_pairs = pd.DataFrame({
    'original_image': ['dog_01.jpg', 'dog_01.jpg', 'car_05.jpg'],
    'augmentation': ['crop + flip', 'grayscale + blur', 'crop + color jitter'],
    'positive_image': ['dog_01_aug1.jpg', 'dog_01_aug2.jpg', 'car_05_aug1.jpg'],
    'cost': ['$0', '$0', '$0'],
    'always_similar': [True, True, True]
})
print(self_sup_pairs.to_string(index=False))
print(f"\nFor ANY number of images: cost = $0, time = seconds")
print(f"No noise: augmented version is ALWAYS similar to original")
print(f"Caveat: augmented images differ from real-world similar images")

## 4. NT-Xent Loss: The Math Behind Contrastive Learning

### The Analogy (for a 12-year-old)

Imagine a classroom quiz:
- Teacher shows you a photo of YOUR dog (query)
- She puts 10 photos on the board: 1 is another photo of your dog (positive), 9 are random animals (negatives)
- You need to pick which one matches
- **Temperature** controls how strict the teacher is: low temperature = "you must be 100% certain", high temperature = "close enough is okay"

### The Math (Step by Step)

NT-Xent stands for **Normalized Temperature-scaled Cross-Entropy** loss.

Given:
- Query embedding: $z_q$
- Positive embedding: $z_+$
- Negative embeddings: $z_1^-, z_2^-, ..., z_{N-1}^-$
- Temperature: $\tau$

**Step 1:** Compute cosine similarities
$$s_+ = \frac{z_q \cdot z_+}{\|z_q\| \|z_+\|}, \quad s_i^- = \frac{z_q \cdot z_i^-}{\|z_q\| \|z_i^-\|}$$

**Step 2:** Apply temperature scaling
$$\text{logits} = [s_+/\tau, \; s_1^-/\tau, \; s_2^-/\tau, \; ...]$$

**Step 3:** Softmax + Cross-entropy (positive should have highest probability)
$$\mathcal{L} = -\log \frac{\exp(s_+/\tau)}{\exp(s_+/\tau) + \sum_{i=1}^{N-1} \exp(s_i^-/\tau)}$$

This is equivalent to cross-entropy where the ground truth label is 0 (first element = positive).

In [ ]:
# ======================================================================
# NT-Xent Loss Implementation (from scratch)
#
# We build it step by step so you can see every computation.
# ======================================================================

class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross-Entropy Loss (NT-Xent).
    
    Used in SimCLR for self-supervised contrastive learning.
    
    For a batch of N images, we get 2N augmented views.
    For each view, its positive is the other augmentation of the same image.
    All other 2(N-1) views are negatives.
    """
    
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z_i, z_j):
        """
        Args:
            z_i: (N, D) embeddings from augmentation 1
            z_j: (N, D) embeddings from augmentation 2
        
        Returns:
            loss: scalar
        """
        N = z_i.shape[0]
        
        # Concatenate: [z_i_1, z_i_2, ..., z_i_N, z_j_1, z_j_2, ..., z_j_N]
        z = torch.cat([z_i, z_j], dim=0)  # (2N, D)
        
        # Compute pairwise cosine similarity matrix
        # sim[a][b] = cosine_sim(z[a], z[b])
        sim = F.cosine_similarity(z.unsqueeze(0), z.unsqueeze(1), dim=2)  # (2N, 2N)
        
        # Scale by temperature
        sim = sim / self.temperature
        
        # Create mask to exclude self-similarity (diagonal)
        mask = torch.eye(2 * N, dtype=torch.bool)
        sim.masked_fill_(mask, float('-inf'))
        
        # For each z_i[k], the positive is z_j[k] (at index N+k)
        # For each z_j[k], the positive is z_i[k] (at index k)
        positive_indices = torch.cat([
            torch.arange(N, 2 * N),  # z_i's positives are in z_j
            torch.arange(0, N)        # z_j's positives are in z_i
        ])
        
        # Cross-entropy loss
        loss = F.cross_entropy(sim, positive_indices)
        
        return loss


# ---- Demo ----
torch.manual_seed(42)

batch_size = 8
embedding_dim = 256

# Simulate embeddings from two augmented views
z_i = F.normalize(torch.randn(batch_size, embedding_dim), dim=1)
z_j = F.normalize(torch.randn(batch_size, embedding_dim), dim=1)

# Make positives more similar (as a trained model would)
z_j = F.normalize(z_i + 0.1 * torch.randn_like(z_i), dim=1)

loss_fn = NTXentLoss(temperature=0.07)
loss = loss_fn(z_i, z_j)

print(f"NT-Xent Loss: {loss.item():.4f}")
print(f"Random guess baseline: -ln(1/{2*batch_size-1}) = {-np.log(1/(2*batch_size-1)):.4f}")
print(f"Perfect model target: 0.0")
print()
print("How it works step by step:")
print(f"  1. Batch of {batch_size} images -> {2*batch_size} augmented views")
print(f"  2. Each view has 1 positive + {2*batch_size-2} negatives")
print(f"  3. Compute {2*batch_size}x{2*batch_size} similarity matrix")
print(f"  4. Scale by temperature (1/{loss_fn.temperature} = {1/loss_fn.temperature:.0f}x amplification)")
print(f"  5. Cross-entropy: push positives up, negatives down")

In [ ]:
# ======================================================================
# Step-by-Step NT-Xent Computation (Manually)
#
# Let's trace through the math with tiny numbers so you can
# see exactly what happens at each step.
# ======================================================================

print("=== Manual NT-Xent Computation (Batch Size = 2) ===")
print()

# Tiny example: 2 images, 3-D embeddings
z_i_manual = torch.tensor([[1.0, 0.0, 0.0],    # Image 1, view 1
                            [0.0, 1.0, 0.0]])    # Image 2, view 1

z_j_manual = torch.tensor([[0.9, 0.1, 0.0],    # Image 1, view 2 (similar to z_i[0])
                            [0.1, 0.9, 0.0]])    # Image 2, view 2 (similar to z_i[1])

# L2 normalize
z_i_manual = F.normalize(z_i_manual, dim=1)
z_j_manual = F.normalize(z_j_manual, dim=1)

print("Embeddings (L2-normalized):")
print(f"  z_i[0] (Image 1, View 1): {z_i_manual[0].tolist()}")
print(f"  z_i[1] (Image 2, View 1): {z_i_manual[1].tolist()}")
print(f"  z_j[0] (Image 1, View 2): {z_j_manual[0].numpy().round(4).tolist()}")
print(f"  z_j[1] (Image 2, View 2): {z_j_manual[1].numpy().round(4).tolist()}")

# Step 1: Concatenate
z_all = torch.cat([z_i_manual, z_j_manual], dim=0)  # (4, 3)
print(f"\nStep 1: Concatenated into {z_all.shape[0]} vectors")

# Step 2: Compute similarity matrix
sim_matrix = z_all @ z_all.T
print(f"\nStep 2: Similarity Matrix (4x4):")
labels = ['I1V1', 'I2V1', 'I1V2', 'I2V2']
print(f"{'':>8}", end='')
for l in labels:
    print(f"{l:>8}", end='')
print()
for i, row in enumerate(sim_matrix):
    print(f"{labels[i]:>8}", end='')
    for val in row:
        print(f"{val.item():>8.4f}", end='')
    print()

# Step 3: Temperature scaling
tau = 0.5  # Using a visible temperature for illustration
scaled_sim = sim_matrix / tau
print(f"\nStep 3: After temperature scaling (tau={tau}):")
print(f"  Similarities are amplified by {1/tau:.0f}x")
print(f"  Positive pair (I1V1, I1V2): {sim_matrix[0,2].item():.4f} -> {scaled_sim[0,2].item():.4f}")
print(f"  Negative pair (I1V1, I2V1): {sim_matrix[0,1].item():.4f} -> {scaled_sim[0,1].item():.4f}")

# Step 4: For z_i[0], compute loss
print(f"\nStep 4: Loss for Image 1, View 1:")
print(f"  Positive: I1V2 (index 2)")
print(f"  Negatives: I2V1 (index 1), I2V2 (index 3)")
print(f"  (Self I1V1 at index 0 is excluded)")

# Manual softmax (excluding self)
logits = torch.tensor([scaled_sim[0,1].item(), scaled_sim[0,2].item(), scaled_sim[0,3].item()])
probs = F.softmax(logits, dim=0)
print(f"\n  Logits: [I2V1={logits[0]:.4f}, I1V2={logits[1]:.4f}, I2V2={logits[2]:.4f}]")
print(f"  Softmax: [I2V1={probs[0]:.4f}, I1V2={probs[1]:.4f}, I2V2={probs[2]:.4f}]")
print(f"  Loss = -log(P(I1V2)) = -log({probs[1]:.4f}) = {-torch.log(probs[1]).item():.4f}")
print(f"\n  The model is {probs[1].item()*100:.1f}% confident the positive is correct!")

## 5. Temperature Parameter: The Sharpness Dial

### The Analogy

Think of temperature like the difficulty setting on a video game:

- **Low temperature (0.01):** "HARD MODE" -- The model must be extremely confident. Even small differences between positive and negative pairs get heavily penalized. The softmax distribution becomes very sharp (almost one-hot).

- **High temperature (1.0):** "EASY MODE" -- The model can be more uncertain. The softmax distribution is flatter, and the model doesn't need to separate positives from negatives as aggressively.

**The sweet spot (0.05-0.1):** Hard enough to learn good representations, but not so hard that training becomes unstable.

In [ ]:
# ======================================================================
# Temperature Exploration: How It Affects the Loss Landscape
# ======================================================================

# Simulate similarity scores: 1 positive (high sim) + 9 negatives (lower sim)
torch.manual_seed(42)
positive_sim = 0.8   # High similarity with positive
negative_sims = torch.tensor([0.3, 0.2, 0.1, 0.4, 0.15, 0.25, 0.05, 0.35, 0.22])
all_sims = torch.cat([torch.tensor([positive_sim]), negative_sims])

temperatures = [0.01, 0.05, 0.07, 0.1, 0.5, 1.0]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

results = []
for idx, tau in enumerate(temperatures):
    scaled = all_sims / tau
    probs = F.softmax(scaled, dim=0)
    loss = -torch.log(probs[0]).item()  # Loss for positive
    
    results.append({'temperature': tau, 'P(positive)': probs[0].item(), 'loss': loss})
    
    # Plot probability distribution
    ax = axes[idx]
    bars = ax.bar(range(10), probs.numpy(), color=['#4CAF50'] + ['#F44336'] * 9,
                  edgecolor='white', linewidth=0.5)
    ax.set_title(f'tau = {tau}\nP(+) = {probs[0].item():.4f}, Loss = {loss:.4f}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Candidate Index')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1.1)
    ax.set_xticks(range(10))
    ax.set_xticklabels(['pos'] + [f'neg{i}' for i in range(1,10)], rotation=45, fontsize=8)

fig.suptitle('Effect of Temperature on Softmax Distribution', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 60)
print(f"{'Temperature':>12} {'P(positive)':>14} {'Loss':>10} {'Interpretation':>25}")
print("=" * 60)
for r in results:
    if r['temperature'] <= 0.01:
        interp = 'Extremely sharp'
    elif r['temperature'] <= 0.07:
        interp = 'Sharp (good default)'
    elif r['temperature'] <= 0.1:
        interp = 'Moderate'
    else:
        interp = 'Too soft'
    print(f"{r['temperature']:>12.2f} {r['P(positive)']:>14.6f} {r['loss']:>10.4f} {interp:>25}")

print("\n--- Interview Talking Point ---")
print("Temperature 0.05-0.1 is the sweet spot for most visual search systems.")
print("Too low: training is unstable (gradients explode).")
print("Too high: model doesn't learn to discriminate well.")
print("The original SimCLR paper uses tau=0.07 as the default.")

## 6. Hard Negative Mining

### The Analogy

Think of studying for a test. If the practice questions are too easy ("Is 2+2=5?"), you don't learn much. But if they're tricky ("Is 17x23 = 381 or 391?"), you learn a LOT more from getting them right.

**Hard negatives** are images that look similar to the query but are actually different. They force the model to learn subtle differences.

```
Easy Negative:  query=golden_retriever, negative=pizza        (Obviously different)
Hard Negative:  query=golden_retriever, negative=labrador     (Looks similar!)
Semi-hard:      query=golden_retriever, negative=poodle       (Same species, different breed)
```

### Three Hard Negative Mining Strategies

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| **In-batch** | Use the hardest negatives within the current batch | Simple, no extra cost | Limited by batch size |
| **Offline mining** | Pre-compute embeddings, find hard negatives for each query | Better negatives | Stale (embeddings change during training) |
| **Semi-hard** | Negatives closer than positive but not TOO close | Stable training | May miss the hardest cases |

In [ ]:
# ======================================================================
# Hard Negative Mining Demo
#
# We'll show how different negative selection strategies affect training.
# ======================================================================

torch.manual_seed(42)
np.random.seed(42)

# Simulate a query embedding and candidates with varying similarity
query = F.normalize(torch.randn(1, 128), dim=1)

# Create negatives at different difficulty levels
n_negatives = 20
negatives = F.normalize(torch.randn(n_negatives, 128), dim=1)

# Compute similarities
similarities = (query @ negatives.T).squeeze()  # (20,)

# Sort by similarity (descending)
sorted_indices = torch.argsort(similarities, descending=True)
sorted_sims = similarities[sorted_indices]

# Categorize negatives
print("All negatives ranked by similarity to query:")
print(f"{'Rank':>6} {'Similarity':>12} {'Difficulty':>15}")
print("-" * 35)
for i, (idx, sim) in enumerate(zip(sorted_indices, sorted_sims)):
    if sim > 0.15:
        difficulty = 'HARD'
    elif sim > 0.0:
        difficulty = 'Semi-hard'
    else:
        difficulty = 'Easy'
    if i < 15:  # Show top 15
        print(f"{i+1:>6} {sim.item():>12.4f} {difficulty:>15}")

print(f"\n--- Mining Strategies ---")
# Strategy 1: Random negatives (baseline)
random_idx = np.random.choice(n_negatives, 5, replace=False)
random_sims = similarities[random_idx]
print(f"\nRandom negatives avg similarity:    {random_sims.mean().item():.4f}")

# Strategy 2: Hard negatives (top-k most similar)
hard_sims = sorted_sims[:5]
print(f"Hard negatives avg similarity:      {hard_sims.mean().item():.4f}")

# Strategy 3: Semi-hard negatives (not too easy, not too hard)
semi_hard_sims = sorted_sims[3:8]
print(f"Semi-hard negatives avg similarity: {semi_hard_sims.mean().item():.4f}")

print("\n--- Why Hard Negatives Matter ---")
print("Random negatives: model easily separates them -> small gradients -> slow learning")
print("Hard negatives: model struggles -> large gradients -> fast learning")
print("Semi-hard: balanced approach, prevents training collapse")

In [ ]:
# ======================================================================
# Training with Hard Negatives vs Random Negatives
#
# We simulate how loss decreases over epochs with different strategies.
# ======================================================================

np.random.seed(42)

epochs = 50

# Simulate loss curves (exponential decay with different rates)
# Hard negatives converge faster but may be noisy
random_neg_loss = 2.3 * np.exp(-0.04 * np.arange(epochs)) + 0.3 + np.random.randn(epochs) * 0.05
semi_hard_loss = 2.3 * np.exp(-0.07 * np.arange(epochs)) + 0.15 + np.random.randn(epochs) * 0.04
hard_neg_loss = 2.3 * np.exp(-0.10 * np.arange(epochs)) + 0.1 + np.random.randn(epochs) * 0.08

fig, ax = plt.subplots(1, 1, figsize=(12, 6))
ax.plot(random_neg_loss, label='Random Negatives', linewidth=2, color='#F44336')
ax.plot(semi_hard_loss, label='Semi-Hard Negatives', linewidth=2, color='#FF9800')
ax.plot(hard_neg_loss, label='Hard Negatives', linewidth=2, color='#4CAF50')

ax.set_xlabel('Epoch', fontsize=14)
ax.set_ylabel('Contrastive Loss', fontsize=14)
ax.set_title('Effect of Negative Mining Strategy on Training', fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 3)

# Annotations
ax.annotate('Hard negatives converge\nfaster but noisier', xy=(15, hard_neg_loss[15]),
            xytext=(25, 1.5), fontsize=10, arrowprops=dict(arrowstyle='->', color='#4CAF50'),
            bbox=dict(boxstyle='round', facecolor='#E8F5E9'))
ax.annotate('Random negatives\nconverge slowly', xy=(40, random_neg_loss[40]),
            xytext=(30, 2.2), fontsize=10, arrowprops=dict(arrowstyle='->', color='#F44336'),
            bbox=dict(boxstyle='round', facecolor='#FFEBEE'))

plt.tight_layout()
plt.show()

print("Key takeaway: Hard negative mining significantly speeds up training.")
print("In practice, use SEMI-HARD negatives for stability.")
print("Pinterest uses hard negative mining in production for their visual search.")

## 7. Pre-trained Model Fine-tuning Workflow

### The Analogy

Think of a pre-trained model like a transfer student who already knows math, science, and reading. You don't need to teach them everything from scratch -- you just need to teach them your school's specific rules and culture.

Similarly, a model pre-trained on ImageNet already knows about edges, textures, shapes, and objects. We just need to **fine-tune** it for our visual search task.

### Fine-tuning Strategy

```
Phase 1: Pre-train (done for us)
    - ImageNet classification (1000 classes, 1.2M images)
    - Model learns general visual features

Phase 2: Self-supervised contrastive pre-training
    - SimCLR/MoCo on our image corpus
    - Freeze backbone, train projection head
    - Then unfreeze and fine-tune everything at lower learning rate

Phase 3: Fine-tune with task-specific data (optional)
    - Use click data or human annotations
    - Very low learning rate (1e-5) to not destroy pre-trained features
```

In [ ]:
# ======================================================================
# Fine-tuning a Pre-trained Model for Visual Search
#
# This is the PRACTICAL workflow you'd use in production.
# We show the code structure (would need real data to actually train).
# ======================================================================

import torchvision.models as models

class VisualSearchFineTuner(nn.Module):
    """
    Fine-tuning workflow for visual search:
    
    1. Load pre-trained ResNet-50 (knows general features)
    2. Replace classification head with projection head
    3. Phase 1: Freeze backbone, train projection head only (fast)
    4. Phase 2: Unfreeze backbone, fine-tune everything (slow, low LR)
    """
    
    def __init__(self, embedding_dim=256):
        super().__init__()
        
        # Load pre-trained backbone
        resnet = models.resnet50(pretrained=False)  # pretrained=True in practice
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Projection head (this is what we train first)
        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Linear(1024, embedding_dim),
        )
        
    def freeze_backbone(self):
        """Phase 1: Only train the projection head."""
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("Backbone FROZEN -- only projection head will train")
        
    def unfreeze_backbone(self):
        """Phase 2: Fine-tune everything."""
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("Backbone UNFROZEN -- everything will train (use low LR!)")
    
    def forward(self, x):
        features = self.backbone(x).flatten(1)  # (B, 2048)
        embedding = self.projection(features)    # (B, embedding_dim)
        embedding = F.normalize(embedding, p=2, dim=1)  # L2 normalize
        return embedding


# ---- Fine-tuning workflow ----
model = VisualSearchFineTuner(embedding_dim=256)

# Phase 1: Train projection head only
model.freeze_backbone()
trainable_phase1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nPhase 1 (Projection head only):")
print(f"  Trainable parameters: {trainable_phase1:,} ({100*trainable_phase1/total_params:.1f}%)")
print(f"  Learning rate: 1e-3")
print(f"  Epochs: 5-10")
print(f"  Why: Fast convergence, don't mess up pre-trained features")

# Phase 2: Fine-tune everything
model.unfreeze_backbone()
trainable_phase2 = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nPhase 2 (Full fine-tuning):")
print(f"  Trainable parameters: {trainable_phase2:,} ({100*trainable_phase2/total_params:.1f}%)")
print(f"  Learning rate: 1e-5 (10-100x lower than Phase 1!)")
print(f"  Epochs: 20-50")
print(f"  Why: Carefully adapt features without destroying them")

# Optimizer setup (differential learning rates)
print("\n--- Staff-Level Trick: Differential Learning Rates ---")
print("Use DIFFERENT learning rates for backbone vs. projection head:")

optimizer = torch.optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 1e-5},   # Low LR for backbone
    {'params': model.projection.parameters(), 'lr': 1e-3},  # Higher LR for projection
])

print(f"  Backbone LR:    1e-5 (pre-trained, don't break it)")
print(f"  Projection LR:  1e-3 (randomly initialized, needs to learn fast)")
print(f"\nThis is called 'discriminative fine-tuning' -- mention it in interviews!")

## 8. Embedding Quality Evaluation

### How Do We Know Our Embeddings Are Good?

Think of it like grading a student's map-drawing skills. A good map should:
1. Put cities that are close in real life close together on the map
2. Put cities that are far apart in real life far apart on the map
3. Group cities in the same country together

Similarly, good embeddings should:
1. Give high similarity to truly similar images
2. Give low similarity to different images  
3. Cluster images of the same category together

### Key Evaluation Metrics

| Metric | What It Measures | How to Compute |
|--------|-----------------|----------------|
| **Recall@K** | How many true neighbors are in top-K | For each query, check if true positives appear in top-K results |
| **Mean Average Precision** | Ranking quality of retrieved results | Average precision across all queries |
| **nDCG** | Ranking quality with graded relevance | Compare DCG to ideal DCG (our primary metric!) |
| **Silhouette Score** | How well categories cluster | Intra-cluster vs inter-cluster distance |
| **Alignment & Uniformity** | Positive pair alignment + uniform distribution | Direct embedding space quality measures |

In [ ]:
# ======================================================================
# Embedding Quality Evaluation Suite
#
# Implements the key metrics for evaluating embedding quality.
# ======================================================================

from sklearn.metrics import silhouette_score

def recall_at_k(embeddings, labels, k=5):
    """
    Recall@K: For each query, what fraction of its true neighbors
    appear in the top-K retrieved results?
    
    Think of it like this: if there are 10 photos of your dog in the
    database, and the top-5 results include 3 of them, Recall@5 = 3/10.
    """
    n = len(embeddings)
    # Compute all pairwise similarities
    sim_matrix = embeddings @ embeddings.T
    
    recalls = []
    for i in range(n):
        # Get similarities (exclude self)
        sims = sim_matrix[i].copy()
        sims[i] = -float('inf')  # exclude self
        
        # Get top-K indices
        top_k_idx = np.argsort(sims)[-k:]
        
        # Count how many have the same label
        same_label = labels[top_k_idx] == labels[i]
        total_same_label = np.sum(labels == labels[i]) - 1  # exclude self
        
        if total_same_label > 0:
            recalls.append(np.sum(same_label) / min(k, total_same_label))
    
    return np.mean(recalls)


def alignment_score(embeddings, labels):
    """
    Alignment: measures how close positive pairs are.
    Lower is better (positive pairs should be very close).
    """
    distances = []
    for label in np.unique(labels):
        mask = labels == label
        group_embs = embeddings[mask]
        # Average pairwise distance within group
        if len(group_embs) > 1:
            sim = group_embs @ group_embs.T
            # Convert similarity to distance
            dist = 2 - 2 * sim  # For normalized vectors
            n_pairs = len(group_embs) * (len(group_embs) - 1)
            distances.append((dist.sum() - np.trace(dist)) / n_pairs)
    return np.mean(distances)


def uniformity_score(embeddings, t=2):
    """
    Uniformity: measures how uniformly distributed embeddings are
    on the unit hypersphere. Lower is better.
    """
    # Pairwise squared distances
    sq_dists = 2 - 2 * (embeddings @ embeddings.T)
    # Uniformity metric
    n = len(embeddings)
    mask = ~np.eye(n, dtype=bool)
    return np.log(np.mean(np.exp(-t * sq_dists[mask])))


# ---- Evaluate our simulated embeddings ----
print("=" * 60)
print("Embedding Quality Evaluation Report")
print("=" * 60)

# Using the embeddings from Section 1
r1 = recall_at_k(all_embeddings, all_labels, k=1)
r5 = recall_at_k(all_embeddings, all_labels, k=5)
r10 = recall_at_k(all_embeddings, all_labels, k=10)
sil = silhouette_score(all_embeddings, all_labels)
align = alignment_score(all_embeddings, all_labels)
uniform = uniformity_score(all_embeddings)

print(f"\nRetrieval Metrics:")
print(f"  Recall@1:  {r1:.4f}  (fraction of queries where top-1 is correct category)")
print(f"  Recall@5:  {r5:.4f}  (fraction of top-5 that are correct category)")
print(f"  Recall@10: {r10:.4f}  (fraction of top-10 that are correct category)")

print(f"\nClustering Quality:")
print(f"  Silhouette Score: {sil:.4f}  (range [-1,1], higher = better clustering)")

print(f"\nEmbedding Space Quality:")
print(f"  Alignment:  {align:.4f}  (lower = positive pairs are closer, good!)")
print(f"  Uniformity: {uniform:.4f}  (lower = more uniform distribution, good!)")

print("\n--- Interview Talking Points ---")
print("1. Use Recall@K for retrieval quality (K=1, 5, 10, 100)")
print("2. Use nDCG for ranking quality (our PRIMARY metric)")
print("3. Use Silhouette Score for clustering quality")
print("4. Alignment + Uniformity are theoretically grounded")
print("   (from the paper 'Understanding Contrastive Learning')")

In [ ]:
# ======================================================================
# Comparing Good vs Bad Embeddings (Visual)
# ======================================================================

np.random.seed(42)

# Good embeddings: tight, well-separated clusters
good_embeddings = []
for i in range(5):
    center = np.random.randn(50) * 3
    points = center + np.random.randn(80, 50) * 0.3  # Tight clusters
    points = points / np.linalg.norm(points, axis=1, keepdims=True)
    good_embeddings.append(points)
good_embeddings = np.vstack(good_embeddings)
good_labels = np.repeat(np.arange(5), 80)

# Bad embeddings: overlapping, messy clusters
bad_embeddings = []
for i in range(5):
    center = np.random.randn(50) * 0.5
    points = center + np.random.randn(80, 50) * 2.0  # Wide, overlapping
    points = points / np.linalg.norm(points, axis=1, keepdims=True)
    bad_embeddings.append(points)
bad_embeddings = np.vstack(bad_embeddings)
bad_labels = np.repeat(np.arange(5), 80)

# t-SNE for both
tsne_good = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(good_embeddings)
tsne_bad = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(bad_embeddings)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for i in range(5):
    mask = good_labels == i
    axes[0].scatter(tsne_good[mask, 0], tsne_good[mask, 1], c=colors[i],
                    label=categories[i], alpha=0.6, s=30)
    mask = bad_labels == i
    axes[1].scatter(tsne_bad[mask, 0], tsne_bad[mask, 1], c=colors[i],
                    label=categories[i], alpha=0.6, s=30)

# Compute metrics for comparison
good_r5 = recall_at_k(good_embeddings, good_labels, k=5)
bad_r5 = recall_at_k(bad_embeddings, bad_labels, k=5)
good_sil = silhouette_score(good_embeddings, good_labels)
bad_sil = silhouette_score(bad_embeddings, bad_labels)

axes[0].set_title(f'Good Embeddings (Well-Trained)\n'
                  f'Recall@5={good_r5:.3f}, Silhouette={good_sil:.3f}',
                  fontsize=13, fontweight='bold', color='#2E7D32')
axes[1].set_title(f'Bad Embeddings (Poorly-Trained)\n'
                  f'Recall@5={bad_r5:.3f}, Silhouette={bad_sil:.3f}',
                  fontsize=13, fontweight='bold', color='#C62828')

for ax in axes:
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

fig.suptitle('Good vs Bad Embedding Spaces', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("Good embeddings: tight clusters, well separated")
print("Bad embeddings: overlapping clusters, can't distinguish categories")
print("The training process (contrastive learning) should move from bad to good!")

## Interview Cheat Sheet: Embeddings & Contrastive Learning

### 30-Second Pitch

> "We use contrastive learning -- specifically SimCLR -- to train a ResNet-50 backbone that maps images to 256-dimensional L2-normalized embeddings. Self-supervised augmentation generates positive pairs at zero cost. We use NT-Xent loss with temperature 0.07 and semi-hard negative mining to learn discriminative representations. We fine-tune from ImageNet pre-trained weights using differential learning rates. We evaluate with nDCG as our primary metric, supplemented by Recall@K and embedding alignment/uniformity scores."

### Key Phrases to Use

| Phrase | When to Use |
|--------|-------------|
| "L2-normalized embeddings on a unit hypersphere" | Describing the embedding space |
| "Cosine similarity equals dot product for normalized vectors" | Explaining similarity computation |
| "Self-supervised positive pairs via SimCLR augmentations" | Training data construction |
| "NT-Xent loss with temperature 0.07" | Loss function |
| "Semi-hard negative mining for stable training" | Training strategy |
| "Discriminative fine-tuning with differential learning rates" | Fine-tuning approach |
| "Alignment and uniformity as embedding quality measures" | Evaluation |

### Common Interview Follow-ups

| Question | Strong Answer |
|----------|---------------|
| Why not Euclidean distance? | Curse of dimensionality -- in high-D, all Euclidean distances converge. Cosine similarity is more discriminative. |
| Why L2 normalize? | Makes cosine sim = dot product (fast computation). Puts all embeddings on same scale. |
| Why not just use ImageNet features? | ImageNet features are for classification, not similarity. Fine-tuning adapts them for our metric (visual similarity). |
| What if augmentation creates false positives? | Use domain-appropriate augmentations. E.g., for fashion, avoid color changes (color matters for clothes). |
| How to handle new categories? | Embeddings generalize! New categories just cluster in new regions. No retraining needed unless quality drops. |

---

*Next notebook: We dive into ANN search and serving architecture -- how to search billions of embeddings in milliseconds!*

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)